In [6]:
import ROOT

In [7]:
file_path = "../root_ML/merged_ML.root"
file = ROOT.TFile(file_path)
tree = file.Get("jetTree")

n_events = tree.GetEntries()
print(f"Number of events: {n_events}")

Number of events: 5737037


In [8]:
# Print branch names and their types as a table
for branch in tree.GetListOfBranches():
    print(f"{branch.GetName():<20} {branch.GetTypeName()}")

jet_pt               vector<float>
jet_eta              vector<float>
jet_phi              vector<float>
jet_mass             vector<float>
jetAK                vector<int>
const_pt             vector<vector<float> >
const_eta            vector<vector<float> >
const_phi            vector<vector<float> >
const_mass           vector<vector<float> >
const_charge         vector<vector<int> >
gen_pt               vector<float>
gen_eta              vector<float>
gen_phi              vector<float>
gen_mass             vector<float>
gen_pdgId            vector<int>
tau_time             vector<vector<double> >
tau_deltaR           vector<vector<double> >
lund_coords_x_sd     vector<vector<double> >
lund_coords_y_sd     vector<vector<double> >
lund_kt_sd           vector<vector<double> >
lund_z_sd            vector<vector<double> >
lund_psi_sd          vector<vector<double> >
lund_delta_sd        vector<vector<double> >
lund_mass_sd         vector<vector<double> >
lund_coords_secondary_x_sd vect

## Plot z1 vs z2 for channel2 = 0, 1, 2, 3

In [9]:
hist_0 = ROOT.TH2F("hist_0", "z1 vs z2 for channel2 = 0", 50, 0.1, 0.5, 50, 0.1, 0.5)
hist_1 = ROOT.TH2F("hist_1", "z1 vs z2 for channel2 = 1", 50, 0.1, 0.5, 50, 0.1, 0.5)
hist_2 = ROOT.TH2F("hist_2", "z1 vs z2 for channel2 = 2", 50, 0.1, 0.5, 50, 0.1, 0.5)
hist_3 = ROOT.TH2F("hist_3", "z1 vs z2 for channel2 = 3", 50, 0.1, 0.5, 50, 0.1, 0.5)

In [10]:
for event_i in range(n_events):
    tree.GetEntry(event_i)
    
    for jet_i in range(len(tree.jet_pt)):
        max_kt_index = tree.lund_max_kt_sd[jet_i]
        max_kt_index2 = tree.lund_max_kt_secondary_sd[jet_i]
        channel2 = tree.lund_secondary_idx_sd[jet_i]

        if max_kt_index < 0 or max_kt_index2 < 0 or channel2 < 0 or channel2 > 3:
            continue
        
        z1 = tree.lund_z_sd[jet_i][max_kt_index]
        z2 = tree.lund_z_secondary_sd[jet_i][max_kt_index2]
        
        if channel2 == 0:
            hist_0.Fill(z1, z2)
        elif channel2 == 1:
            hist_1.Fill(z1, z2)
        elif channel2 == 2:
            hist_2.Fill(z1, z2)
        elif channel2 == 3:
            hist_3.Fill(z1, z2)

In [11]:
canvases = []

hist_total = ROOT.TH2F(
    "hist_total",
    "z1 vs z2 for all channel2 values",
    50, 0.1, 0.5,
    50, 0.1, 0.5
)

hist_list = [hist_0, hist_1, hist_2, hist_3]

for hist_i, hist in enumerate(hist_list):
    hist_total.Add(hist)

hist_list_all = [hist_0, hist_1, hist_2, hist_3, hist_total]

for hist_i, hist in enumerate(hist_list_all):
    canvas = ROOT.TCanvas(
        f"canvas_{hist_i}",
        f"z1 vs z2 {hist_i}",
        800,
        600
    )
    canvases.append(canvas)  # important: keep canvases alive

    hist.SetStats(0)
    hist.SetTitle(
        f"z1 vs z2 for channel2 = {hist_i}"
        if hist_i < 4
        else "z1 vs z2 for all channel2 values"
    )
    hist.GetXaxis().SetTitle("z1")
    hist.GetYaxis().SetTitle("z2")
    hist.GetZaxis().SetTitle("Entries")

    hist.Draw("COLZ")
    canvas.Draw()

In [21]:
# Ratio channel2 = x / channel2 = all_other_channels
canvases_ratio = []
hist_ratio_list = []
hist_ratio_0 = ROOT.TH2F("hist_ratio_0", "z1 vs z2 ratio for channel2 = 0 / all_other_channels", 50, 0.1, 0.5, 50, 0.1, 0.5)
hist_ratio_1 = ROOT.TH2F("hist_ratio_1", "z1 vs z2 ratio for channel2 = 1 / all_other_channels", 50, 0.1, 0.5, 50, 0.1, 0.5)
hist_ratio_2 = ROOT.TH2F("hist_ratio_2", "z1 vs z2 ratio for channel2 = 2 / all_other_channels", 50, 0.1, 0.5, 50, 0.1, 0.5)
hist_ratio_3 = ROOT.TH2F("hist_ratio_3", "z1 vs z2 ratio for channel2 = 3 / all_other_channels", 50, 0.1, 0.5, 50, 0.1, 0.5)

for hist_i, hist in enumerate(hist_list):
    hist_ratio = hist.Clone(f"hist_ratio_{hist_i}")
    hist_ratio.Divide(hist_total)
    hist_ratio_list.append(hist_ratio)

    canvas_ratio = ROOT.TCanvas(
        f"canvas_ratio_{hist_i}",
        f"z1 vs z2 ratio {hist_i}",
        800,
        600
    )
    canvases_ratio.append(canvas_ratio)  # important: keep canvases alive

    hist_ratio.SetStats(0)
    hist_ratio.SetTitle(
        f"z1 vs z2 ratio for channel2 = {hist_i} / all_other_channels"
    )
    hist_ratio.GetXaxis().SetTitle("z1")
    hist_ratio.GetYaxis().SetTitle("z2")
    hist_ratio.GetZaxis().SetTitle("Ratio")

    hist_ratio.Draw("COLZ")
    canvas_ratio.Draw()

Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_0 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_0 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_1 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_1 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_2 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_2 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_3 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_ratio_3 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_ratio_0
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_ratio_1
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_ratio_2
Warning in <TCanvas::Constructor>:

## Test different z1 and z2 zones dpsi12 distribution

In [22]:
hist_dpsi12_0 = ROOT.TH1F("hist_dpsi12_0", "dpsi12 for channel2 = 0", 50, 0, 3.14)
hist_dpsi12_1 = ROOT.TH1F("hist_dpsi12_1", "dpsi12 for channel2 = 1", 50, 0, 3.14)
hist_dpsi12_2 = ROOT.TH1F("hist_dpsi12_2", "dpsi12 for channel2 = 2", 50, 0, 3.14)
hist_dpsi12_3 = ROOT.TH1F("hist_dpsi12_3", "dpsi12 for channel2 = 3", 50, 0, 3.14)
hist_dpsi12_total = ROOT.TH1F("hist_dpsi12_total", "dpsi12 for all channel2 values", 50, 0, 3.14)

z1_min = 0.2
z1_max = 0.4
z2_min = 0.3
z2_max = 0.5

for event_i in range(n_events):
    tree.GetEntry(event_i)
    
    for jet_i in range(len(tree.jet_pt)):
        max_kt_index = tree.lund_max_kt_sd[jet_i]
        max_kt_index2 = tree.lund_max_kt_secondary_sd[jet_i]
        channel2 = tree.lund_secondary_idx_sd[jet_i]

        if max_kt_index < 0 or max_kt_index2 < 0 or channel2 < 0 or channel2 > 3:
            continue
        
        z1 = tree.lund_z_sd[jet_i][max_kt_index]
        z2 = tree.lund_z_secondary_sd[jet_i][max_kt_index2]

        if z1_min <= z1 <= z1_max and z2_min <= z2 <= z2_max:
            dpsi12 = tree.lund_psi12_sd[jet_i]
            
            if channel2 == 0:
                hist_dpsi12_0.Fill(dpsi12)
            elif channel2 == 1:
                hist_dpsi12_1.Fill(dpsi12)
            elif channel2 == 2:
                hist_dpsi12_2.Fill(dpsi12)
            elif channel2 == 3:
                hist_dpsi12_3.Fill(dpsi12)
            
            hist_dpsi12_total.Fill(dpsi12)

Warning in <TFile::Append>: Replacing existing TH1: hist_dpsi12_0 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_dpsi12_1 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_dpsi12_2 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_dpsi12_3 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist_dpsi12_total (Potential memory leak).


In [23]:
canvases = []
hist_list_all = [hist_dpsi12_0, hist_dpsi12_1, hist_dpsi12_2, hist_dpsi12_3, hist_dpsi12_total]

for hist_i, hist in enumerate(hist_list_all):
    canvas = ROOT.TCanvas(
        f"canvas_{hist_i}",
        f"dpsi12 {hist_i}",
        800,
        600
    )
    canvases.append(canvas)  # important: keep canvases alive

    hist.SetStats(0)
    hist.SetTitle(f"dpsi12 for channel2 = {hist_i}" if hist_i < 4 else "dpsi12 for all channel2 values")
    hist.GetXaxis().SetTitle("dpsi12")
    hist.GetYaxis().SetTitle("Entries")

    hist.Draw("HIST")
    canvas.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_0
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_1
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_2
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_3
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_4


In [26]:
hist_dpsi12_0.SetFillColor(ROOT.kRed)
hist_dpsi12_1.SetFillColor(ROOT.kBlue)
hist_dpsi12_2.SetFillColor(ROOT.kGreen)
hist_dpsi12_3.SetFillColor(ROOT.kMagenta)

hs = ROOT.THStack("hs", "dpsi12 for all channel2 values")
hs.Add(hist_dpsi12_0)
hs.Add(hist_dpsi12_1)
hs.Add(hist_dpsi12_2)
hs.Add(hist_dpsi12_3)

legend = ROOT.TLegend(0.7, 0.7, 0.9, 0.9)
legend.AddEntry(hist_dpsi12_0, "channel2 = 0", "f")
legend.AddEntry(hist_dpsi12_1, "channel2 = 1", "f")
legend.AddEntry(hist_dpsi12_2, "channel2 = 2", "f")
legend.AddEntry(hist_dpsi12_3, "channel2 = 3", "f")
legend.SetBorderSize(0)
legend.SetFillStyle(0)

canvas_hs = ROOT.TCanvas("canvas_hs", "dpsi12 for all channel2 values", 800, 600)
hs.SetTitle("dpsi12 for all channel2 values")
hs.Draw("HIST")
legend.Draw()
canvas_hs.Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_hs
